# L3 Pandas II: Conditionals & Modification

- Conditional Selection

- Modification

- Some Useful Functions

- Custom Sorts

In [1]:
# This code pulls census data and loads it into a DataFrame
# We won't cover it explicitly in this class, but you are welcome to explore it on your own
import pandas as pd
import numpy as np
import urllib.request
import os.path
import zipfile

data_url = "https://www.ssa.gov/oact/babynames/state/namesbystate.zip"
local_filename = "data/babynamesbystate.zip"
if not os.path.exists(local_filename): # If the data exists don't download again
    with urllib.request.urlopen(data_url) as resp, open(local_filename, 'wb') as f:
        f.write(resp.read())

zf = zipfile.ZipFile(local_filename, 'r')

ca_name = 'STATE.CA.TXT'
field_names = ['State', 'Sex', 'Year', 'Name', 'Count']
with zf.open(ca_name) as fh:
    babynames = pd.read_csv(fh, header=None, names=field_names)

babynames.head()

HTTPError: HTTP Error 403: Forbidden

It may error because the server rejects Python program to access it

**We just download the file manually for now!** 😬

In [2]:
import pandas as pd
import numpy as np

local_filename = "CA.TXT"

field_names = ["State", "Sex", "Year", "Name", "Count"]

babynames = pd.read_csv(
    local_filename,
    header=None,
    names=field_names
)

babynames.head()

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
1,CA,F,1910,Helen,239
2,CA,F,1910,Dorothy,220
3,CA,F,1910,Margaret,163
4,CA,F,1910,Frances,134


## 1 Conditional Selection

### Logical Conditional

- We want to use `True` or `False` to filter entries

**We can use `[]` with booleans**

In [8]:
# Ask yourself: why is :9 is the correct slice to select the first 10 rows?
babynames_first_10_rows = babynames.loc[:9, :]

# .loc is a label-based indexer, so it is inclusive
# .iloc is a position-based indexer, so it is exclusive

# Notice how we have exactly 10 elements in our boolean array argument
babynames_first_10_rows[[True, False, True, False, True, False, True, False, True, False]]

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
2,CA,F,1910,Dorothy,220
4,CA,F,1910,Frances,134
6,CA,F,1910,Evelyn,126
8,CA,F,1910,Virginia,101


**We can also use `.loc` with booleans**

In [9]:
babynames_first_10_rows.iloc[[True, False, True, False, True, False, True, False, True, False]]

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
2,CA,F,1910,Dorothy,220
4,CA,F,1910,Frances,134
6,CA,F,1910,Evelyn,126
8,CA,F,1910,Virginia,101


- We don't need to mannually enter booleans, we can use a logical condition to generate a boolean **array**

- Say we wanna filter out all names with `F` sex

In [11]:
# Logical conditions to generate a boolean array
logical_operator = (babynames['Sex'] == 'F')
logical_operator

0          True
1          True
2          True
3          True
4          True
          ...  
426816    False
426817    False
426818    False
426819    False
426820    False
Name: Sex, Length: 426821, dtype: bool

In [12]:
# Use this boolean array to filter the DataFrame
babynames[logical_operator].head()

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
1,CA,F,1910,Helen,239
2,CA,F,1910,Dorothy,220
3,CA,F,1910,Margaret,163
4,CA,F,1910,Frances,134


- We can also use `.loc`

In [13]:
babynames.loc[babynames["Sex"] == "F"].head()

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
1,CA,F,1910,Helen,239
2,CA,F,1910,Dorothy,220
3,CA,F,1910,Margaret,163
4,CA,F,1910,Frances,134


<div style="background-color:#eab30826; padding:12px 16px; border-left:5px solid #eab308; width:100%; max-width:100%; box-sizing:border-box; overflow-x:auto;">

<strong>Key idea:</strong> **Logical Condition**

`logical_operator = (Conditions)`

`babynames.loc[Conditions]`

</div>

- In fact, `[]` can take a boolean **array, Series, list**

Conditions support bitwise operators

| Symbol | Usage | Meaning |
| --- | --- | --- |
| ~ | ~p | Returns negation of p |
| \| | p \| q | p OR q |
| & | p & q | p AND q |
| ^ | p ^ q | p XOR q (exclusive or) |

- Use `()` to indicate the order of operation

In [15]:
babynames[(babynames["Sex"] == "F") & (babynames["Year"] < 2000)].head()

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
1,CA,F,1910,Helen,239
2,CA,F,1910,Dorothy,220
3,CA,F,1910,Margaret,163
4,CA,F,1910,Frances,134


In [14]:
# This line of code will raise a ValueError
babynames[(babynames["Sex"] == "F") and (babynames["Year"] < 2000)].head()

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

<div style="background-color:#ef444426; padding:12px 16px; border-left:5px solid #ef4444; width:100%; max-width:100%; box-sizing:border-box; overflow-x:auto;">

<strong>Warning:</strong> 

- Note that we’re working with Series, so using `and` in place of `&`, or `or` in place `|` will **error**

</div>

- There are many alternatives for constructing boolean filters

### `.isin`

- `.isin` evaluates if the values in a **Series** are also in another given sequence (list, array, Series)

In [16]:
names = ["Bella", "Alex", "Narges", "Lisa"] # Given sequence
babynames["Name"].isin(names).head()

0    False
1    False
2    False
3    False
4    False
Name: Name, dtype: bool

In [17]:
# Filter out the rows with the given names
babynames[babynames["Name"].isin(names)].head()

,State,Sex,Year,Name,Count
6289,CA,F,1923,Bella,5
7512,CA,F,1925,Bella,8
12368,CA,F,1932,Lisa,5
14741,CA,F,1936,Lisa,8
17084,CA,F,1939,Lisa,5


### `str.startswith`

- The function `str.startswith` checks if string values in a Series starts with a perticular string (But here we just use one character)

In [19]:
# Extracting names that begin with the letter "N"
babynames[babynames["Name"].str.startswith("N")].head()

,State,Sex,Year,Name,Count
76,CA,F,1910,Norma,23
83,CA,F,1910,Nellie,20
127,CA,F,1910,Nina,11
198,CA,F,1910,Nora,6
310,CA,F,1911,Nellie,23


## 2 Modification

- Sometimes we need to make some changes to our columns

### Adding

- By defining a new col with its values

In [20]:
# Create a Series of the length of each name. 
babyname_lengths = babynames["Name"].str.len()

# Add a column named "name_lengths" that includes the length of each name
babynames["name_lengths"] = babyname_lengths
babynames.head(5)

,State,Sex,Year,Name,Count,name_lengths
0,CA,F,1910,Mary,295,4
1,CA,F,1910,Helen,239,5
2,CA,F,1910,Dorothy,220,7
3,CA,F,1910,Margaret,163,8
4,CA,F,1910,Frances,134,7


### Updating

- By redefining the existing col with new values

In [21]:
# Modify the “name_lengths” column to be one less than its original value
babynames["name_lengths"] = babynames["name_lengths"] - 1
babynames.head()

,State,Sex,Year,Name,Count,name_lengths
0,CA,F,1910,Mary,295,3
1,CA,F,1910,Helen,239,4
2,CA,F,1910,Dorothy,220,6
3,CA,F,1910,Margaret,163,7
4,CA,F,1910,Frances,134,6


<div style="background-color:#eab30826; padding:12px 16px; border-left:5px solid #eab308; width:100%; max-width:100%; box-sizing:border-box; overflow-x:auto;">

<strong>Key idea:</strong> 

- Assignment will change the initial df

- But the table methods (operations) **usually** won't

</div>

### Renaming `.rename()`

- It takes in **a dictionary** that maps old column names to their new ones

- `.rename(columns={"old_name": "new_name"})`

In [22]:
# Rename “name_lengths” to “Length”
babynames = babynames.rename(columns={"name_lengths":"Length"})
babynames.head()

,State,Sex,Year,Name,Count,Length
0,CA,F,1910,Mary,295,3
1,CA,F,1910,Helen,239,4
2,CA,F,1910,Dorothy,220,6
3,CA,F,1910,Margaret,163,7
4,CA,F,1910,Frances,134,6


- We reassign here, the reason is if we don't do it, it won't change the original df. See more below

### Removing `.drop()`

- Use the `axis` parameter to specify whether a column or row should be dropped

- If not specified, it assumes row drop (in this case, you need to pass into row **index**)

In [23]:
# Drop our new "Length" column from the DataFrame
babynames = babynames.drop("Length", axis="columns")
babynames.head(5)

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
1,CA,F,1910,Helen,239
2,CA,F,1910,Dorothy,220
3,CA,F,1910,Margaret,163
4,CA,F,1910,Frances,134


<div style="background-color:#8b5cf630; padding:12px 16px; border-left:5px solid #8b5cf6; width:100%; max-width:100%; box-sizing:border-box; overflow-x:auto;">

<strong>Question:</strong> **Why we reassign?**

- we re-assigned `babynames` to the result of `babynames.drop(...)`

- Pandas table **operations USUALLY do not occur in-place**

- Calling `df.drop(...)` will **output a copy** of df with the row/column of interest removed **without modifying the original df table**

</div>

In [24]:
# This creates a copy of `babynames` and removes the column "Name"...
babynames.drop("Name", axis="columns")

# ...but the original `babynames` is unchanged! 
# Notice that the "Name" column is still present
babynames.head(5)

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
1,CA,F,1910,Helen,239
2,CA,F,1910,Dorothy,220
3,CA,F,1910,Margaret,163
4,CA,F,1910,Frances,134


## 3 Useful Utility Functions

- NumPy and built-in function support

- `.shape`

- `.size`

- `.describe()`

- `.sample()`

- `.value_counts()`

- `.unique()`

- `.sort_values()`

[Pandas documentation](https://pandas.pydata.org/docs/reference/index.html)

### Numpy

In [25]:
# Pull out the number of babies named Yash each year
yash_count = babynames[babynames["Name"] == "Yash"]["Count"]
# First filter out the rows with name "Yash"
# Then select the col "Count"
yash_count.head()

342722     8
345010     9
347287    11
349669    12
352283    10
Name: Count, dtype: int64

In [26]:
# Average number of babies named Yash each year
np.mean(yash_count)

np.float64(16.70967741935484)

In [27]:
# Max number of babies named Yash born in any one year
np.max(yash_count)

29

### `.shape` & `.size`

In [28]:
# Return the shape of the DataFrame, in the format (num_rows, num_columns)
babynames.shape

(426821, 5)

In [ ]:
# Return the size of the DataFrame, equal to num_rows * num_columns
# Total elements (entries) in the DataFrame
babynames.size

2134105

### `.describe()`

For numeral data

In [30]:
babynames.describe()

,Year,Count
count,426821.000000,426821.000000
mean,1987.468527,78.361934
std,27.561258,288.801683
min,1910.000000,5.000000
25%,1970.000000,7.000000
50%,1993.000000,13.000000
75%,2010.000000,38.000000
max,2025.000000,8264.000000


For categorical data

In [31]:
babynames["Sex"].describe()

count     426821
unique         2
top            F
freq      250430
Name: Sex, dtype: object

### `.sample()`

- It let us **quickly select random entries** (a row if called from a DataFrame, or a value if called from a Series)

- By default, it selects without replacement. Pass `replace=True` to sample with replacement (i.e. put the entries back after selection)

In [36]:
# Sample a single row
babynames.sample()

# Run it multiple times to see a different row each time

,State,Sex,Year,Name,Count
281060,CA,M,1959,James,5025


- Can be chained with other methods and operators (`iloc`, etc.)

In [37]:
# Randomly sample 4 names from the year 2000, with replacement, and select all columns after column 2
babynames[babynames["Year"] == 2000].sample(4, replace = True).iloc[:, 2:]

,Year,Name,Count
150012,2000,Brianne,32
150186,2000,Gemma,25
152657,2000,Mayte,5
150143,2000,Alysia,26


### `.value_counts()`

- `Series.value_counts()` counts the number of occurrence of each unique value in a **Series** (i.e. compute the "frequency" for one column)

In [38]:
babynames["Name"].value_counts().head()

Name
Jean         229
Francis      227
Jessie       223
Guadalupe    222
Marion       216
Name: count, dtype: int64

### `.unique()`

- It gives an **array** of all unique values

In [39]:
babynames["Name"].unique()

array(['Mary', 'Helen', 'Dorothy', ..., 'Yeshaya', 'Zaqueo', 'Zohran'],
      shape=(21031,), dtype=object)

### `.sort_values()`

In [ ]:
# Sort the df according to the "Count" column from highest to lowest
babynames.sort_values(by="Count", ascending=False).head()

,State,Sex,Year,Name,Count
278930,CA,M,1957,Michael,8264
277906,CA,M,1956,Michael,8256
328279,CA,M,1990,Michael,8248
292738,CA,M,1969,Michael,8246
294034,CA,M,1970,Michael,8198


- When it is called on a **Series**, we don't need to specify the column


In [41]:
# Sort the "Name" Series alphabetically
babynames["Name"].sort_values(ascending=True).head()

380022      Aadan
394909      Aadan
376901      Aadan
409135    Aadarsh
404988      Aaden
Name: Name, dtype: object

## 4 *Custom Sorts (Exer.)

Assume we want to **find the longest baby names** and sort our data accordingly

In [52]:
import pandas as pd
import numpy as np

local_filename = "CA.TXT"

field_names = ["State", "Sex", "Year", "Name", "Count"]

babynames = pd.read_csv(
    local_filename,
    header=None,
    names=field_names
)

babynames.tail(10)

,State,Sex,Year,Name,Count
426811,CA,M,2025,Yulian,5
426812,CA,M,2025,Zackery,5
426813,CA,M,2025,Zai,5
426814,CA,M,2025,Zakari,5
426815,CA,M,2025,Zakariah,5
426816,CA,M,2025,Zaqueo,5
426817,CA,M,2025,Zayed,5
426818,CA,M,2025,Ziyad,5
426819,CA,M,2025,Zohran,5
426820,CA,M,2025,Zuko,5


### Approach 1: Temporary column

- Create a col: `"length of name"` then sort it

In [53]:
# Create a Series of the length of each name
babyname_lengths = babynames["Name"].str.len()

# Add a column named "name_lengths" that includes the length of each name
babynames["name_lengths"] = babyname_lengths
babynames.head(5)

,State,Sex,Year,Name,Count,name_lengths
0,CA,F,1910,Mary,295,4
1,CA,F,1910,Helen,239,5
2,CA,F,1910,Dorothy,220,7
3,CA,F,1910,Margaret,163,8
4,CA,F,1910,Frances,134,7


In [54]:
# Sort the temporary col
babynames = babynames.sort_values(by="name_lengths", ascending=False)
babynames.head(5)

,State,Sex,Year,Name,Count,name_lengths
350368,CA,M,1998,Franciscojavier,6,15
324870,CA,M,1988,Franciscojavier,10,15
323436,CA,M,1987,Franciscojavier,5,15
102514,CA,F,1986,Mariadelosangel,5,15
332684,CA,M,1991,Ryanchristopher,7,15


In [55]:
# Drop the tempoary col
babynames = babynames.drop("name_lengths", axis="columns")
babynames.head(5)

,State,Sex,Year,Name,Count
350368,CA,M,1998,Franciscojavier,6
324870,CA,M,1988,Franciscojavier,10
323436,CA,M,1987,Franciscojavier,5
102514,CA,F,1986,Mariadelosangel,5
332684,CA,M,1991,Ryanchristopher,7


### Approach 2: Sorting using the `key` Argument

In [56]:
# Pass a function to key, values will be sent to func first then be sorted
babynames.sort_values("Name", key=lambda x: x.str.len(), ascending=False).head()

,State,Sex,Year,Name,Count
350368,CA,M,1998,Franciscojavier,6
338253,CA,M,1993,Johnchristopher,5
324870,CA,M,1988,Franciscojavier,10
348198,CA,M,1997,Franciscojavier,5
332805,CA,M,1991,Franciscojavier,6


### Approach 3: Sorting using the `map` Function

- **New requirement**: Say we want to sort the babynames table by the number of "dr"'s and "ea"'s in each "Name"

In [57]:
# First, define a function to count the number of times "dr" or "ea" appear in each name
def dr_ea_count(string):
    return string.count('dr') + string.count('ea')

# Then, use `map` to apply `dr_ea_count` to each name in the "Name" column
babynames["dr_ea_count"] = babynames["Name"].map(dr_ea_count)

# Sort the DataFrame by the new "dr_ea_count" column so we can see our handiwork
babynames = babynames.sort_values(by="dr_ea_count", ascending=False)
babynames.head()

,State,Sex,Year,Name,Count,dr_ea_count
131045,CA,F,1994,Leandrea,5,3
319022,CA,M,1985,Deandrea,6,3
108742,CA,F,1988,Deandrea,5,3
115970,CA,F,1990,Deandrea,5,3
101985,CA,F,1986,Deandrea,6,3


In [58]:
# Drop the `dr_ea_count` column
babynames = babynames.drop("dr_ea_count", axis = 'columns')
babynames.head(5)

,State,Sex,Year,Name,Count
131045,CA,F,1994,Leandrea,5
319022,CA,M,1985,Deandrea,6
108742,CA,F,1988,Deandrea,5
115970,CA,F,1990,Deandrea,5
101985,CA,F,1986,Deandrea,6


---
[Reference](https://ds100.org/course-notes/pandas-2/)